# AMML Ranking Competition 2026 — solution

**Preferred runtime:** Kaggle Notebook **GPU** (~30h/week).  
**Also runs:** local CPU.  
**Metric:** NDCG@10  

Pipeline: features → CatBoost YetiRank → fine-tune CrossEncoder on competition pairs → within-query blend.

No external labeled IR datasets.

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
print('platform:', platform.platform())
print('python:', sys.version)
print('on_kaggle:', ON_KAGGLE)

if ON_KAGGLE:
    DATA = Path('/kaggle/input/amml-ranking-competition-2026')
    OUT = Path('/kaggle/working')
    MODELS = OUT / 'models'
    # Prefer code dataset; fall back to git clone
    CODE_DS = Path('/kaggle/input/amml-ranking-2026-code')
    if (CODE_DS / 'src').exists():
        ROOT = CODE_DS
    else:
        ROOT = OUT / 'amml-ranking-competition-2026'
        if not ROOT.exists():
            subprocess.check_call([
                'git', 'clone', '--depth', '1',
                'https://github.com/VenistorAlishi/amml-ranking-competition-2026.git',
                str(ROOT),
            ])
    # Do NOT pip-install catboost on py3.13 (no wheel → build fails).
    # Kaggle images already ship catboost / torch / transformers.
    try:
        import rank_bm25  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25==0.2.2'])
    import catboost, torch, transformers
    print('catboost', catboost.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
else:
    ROOT = Path('.').resolve()
    DATA = ROOT / 'data'
    OUT = ROOT / 'results'
    MODELS = ROOT / 'models'

sys.path.insert(0, str(ROOT))
OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
print('ROOT', ROOT)
print('DATA', DATA)
assert (DATA / 'train.csv').exists(), list(DATA.parent.glob('*')) if ON_KAGGLE else DATA

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.preprocessing import prepare_frame
from src.features import assemble_features
from src.train import cross_validate, train_full, predict, get_feature_cols
from src.ce_train import train_cross_encoder, predict_cross_encoder
from src.metrics import mean_ndcg_at_k
from src.pipeline import blend_frames

train = prepare_frame(pd.read_csv(DATA / 'train.csv'))
test = prepare_frame(pd.read_csv(DATA / 'test.csv'))
print(train.shape, test.shape, 'pos_rate', float(train.relevance.mean()))

In [ ]:
print('Building features...')
feat_tr, feat_te = assemble_features(train, test)
labels = train['relevance']
print('feat_tr', feat_tr.shape)

In [ ]:
print('CatBoost YetiRank GroupKFold CV...')
cv_cb, oof_cb = cross_validate(feat_tr, labels, n_splits=5)
model = train_full(feat_tr, labels, MODELS, best_iterations=1500)
feature_cols = get_feature_cols(feat_tr.assign(relevance=labels))
cb_te = predict(model, feat_te, feature_cols)
print('CatBoost CV NDCG@10', cv_cb)

# Always save CB-only submission (safe baseline)
sub_cb = test[['id']].copy()
sub_cb['prediction'] = cb_te
sub_cb.to_csv(OUT / 'submission_cb_only.csv', index=False)
print('wrote', OUT / 'submission_cb_only.csv')

In [ ]:
ce_epochs = 3 if torch.cuda.is_available() else 1
ce_bs = 64 if torch.cuda.is_available() else 16
print(f'Fine-tuning CE rubert-tiny2 epochs={ce_epochs} bs={ce_bs}')
arts = train_cross_encoder(
    train, MODELS,
    model_name='cointegrated/rubert-tiny2',
    epochs=ce_epochs,
    batch_size=ce_bs,
    max_len=192,
    lr=2e-5,
)
ce_tr = predict_cross_encoder(arts, train, batch_size=128 if torch.cuda.is_available() else 32)
ce_te = predict_cross_encoder(arts, test, batch_size=128 if torch.cuda.is_available() else 32)
print('CE mean/std', float(ce_tr.mean()), float(ce_tr.std()))

In [ ]:
# CE train scores are in-sample → keep weight small; default to CB-only if boost looks fake
best_w, best_adj = 0.0, cv_cb
for w in [0.0, 0.1, 0.15, 0.2, 0.25]:
    pred = blend_frames(train['query_id'].to_numpy(), [oof_cb, ce_tr], [1 - w, w])
    scored = train[['id', 'query_id', 'relevance']].copy()
    scored['prediction'] = pred
    raw = mean_ndcg_at_k(scored)
    adj = raw if w == 0 else min(raw, cv_cb + 0.015)
    print(f'w_ce={w:.2f} raw={raw:.6f} adj={adj:.6f}')
    if adj > best_adj:
        best_adj, best_w = adj, w

print('selected w_ce', best_w)
te_pred = blend_frames(test['query_id'].to_numpy(), [cb_te, ce_te], [1 - best_w, best_w])
sub = test[['id']].copy()
sub['prediction'] = te_pred
sub.to_csv(OUT / 'submission.csv', index=False)
print('wrote', OUT / 'submission.csv', 'rows', len(sub))
sub.head()

### Submit
On Kaggle: **Submit** `/kaggle/working/submission.csv` (and optionally `submission_cb_only.csv`) to the competition from the notebook output panel.